# Proyek Klasifikasi Gambar: Geometric Shapes

- **Nama:** Muslichin
- **Email:** muslichin.ach@gmail.com
- **ID Dicoding:** muslchn

Notebook ini membangun model klasifikasi gambar untuk tiga kelas geometri: `circle`, `square`, dan `triangle`. Dataset dibuat secara reproducible dari kode Python dengan total 12.000 gambar dan resolusi asli yang bervariasi.

## Import Semua Packages/Library yang Digunakan

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from submission import (
    CLASS_NAMES,
    DATASET_DIR,
    SPLIT_DIR,
    SUBMISSION_DIR,
    build_model,
    export_models,
    generate_dataset,
    load_datasets,
    plot_history,
    print_images_resolution,
    run_tflite_inference,
    set_seed,
    split_dataset,
)

set_seed()
print(tf.__version__)

## Data Preparation

### Data Loading

Dataset dibuat dari kode agar proyek dapat dijalankan ulang tanpa unduhan eksternal. Setiap kelas memiliki 4.000 gambar, sehingga total dataset adalah 12.000 gambar.

In [ ]:
generate_dataset()
print_images_resolution(DATASET_DIR)

### Data Preprocessing

#### Split Dataset

Dataset dibagi menjadi train, validation, dan test set dengan rasio 70:15:15. Augmentation hanya diterapkan pada pipeline training di dalam model.

In [ ]:
split_dataset()
train_ds, validation_ds, test_ds = load_datasets()
train_ds, validation_ds, test_ds

## Modelling

Model menggunakan `Sequential`, beberapa layer `Conv2D`, dan `MaxPooling2D`. Callback digunakan untuk menyimpan model terbaik, menghentikan training saat performa validasi tidak membaik, dan menurunkan learning rate saat loss validasi stagnan.

In [ ]:
model = build_model(num_classes=len(CLASS_NAMES))
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(SUBMISSION_DIR / "best_model.keras"),
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    callbacks=callbacks,
)

## Evaluasi dan Visualisasi

In [ ]:
plot_history(history)

train_loss, train_accuracy = model.evaluate(train_ds, verbose=0)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Testing accuracy: {test_accuracy:.4f}")
print(f"Training loss: {train_loss:.4f}")
print(f"Testing loss: {test_loss:.4f}")

## Konversi Model

Model disimpan ke format SavedModel, TensorFlow Lite, dan TensorFlow.js.

In [ ]:
export_models(model)

## Inference

Inference dilakukan menggunakan model TensorFlow Lite yang sudah dikonversi.

In [ ]:
run_tflite_inference()